# SentryMesh Guardian: Parameter-Efficient Fine-Tuning with XLM-RoBERTa & LoRA

This notebook provides the **optional deep fine-tuning stage** for SentryMesh Guardian once the reviewed dataset scale warrants sequence-level transformer parameter updates.

### ⚠️ Methodological Guardrails:
1. **Never Train from Scratch**: We use `FacebookAI/xlm-roberta-base` as the foundation multilingual transformer.
2. **PEFT / LoRA Only**: Adapts low-rank matrices (`r=16, lora_alpha=32, target_modules=["query", "value"]`) to prevent catastrophic forgetting and minimize GPU memory.
3. **Regularization Safeguards**: Early stopping on validation loss, dropout rate 0.1, weight decay 0.01.
4. **Production Architecture**: The MiniLM + LogisticRegression baseline remains the resilient, low-latency production fallback.

In [ ]:
!pip install -q transformers peft accelerate datasets torch evaluate
import torch
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

## 1. Load SentryMesh Multi-Class Dataset

In [ ]:
import json
import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

with open("../models/label_mapping.json", "r") as f:
    mapping = json.load(f)
label2id = mapping["label_to_id"]
id2label = {int(k): v for k, v in mapping["id_to_label"].items()}

train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})
print("Dataset prepared for tokenization.")

## 2. Tokenize with XLM-RoBERTa

In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
    return tokenizer(examples["normalized_text"], truncation=True, max_length=256)

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

## 3. Attach LoRA Adapter

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

lora_model = get_peft_model(base_model, peft_config)
lora_model.print_trainable_parameters()

## 4. Training Arguments with Overfitting Safeguards

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../models/xlmr_lora_checkpoint",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    fp16=torch.cuda.is_available()
)
print("Training configuration ready.")